# PanGBank Tutorial: AMR Gene Analysis in *Acinetobacter baumannii*

***Acinetobacter baumannii*** is a nosocomial opportunistic pathogen ranked by the WHO as a critical-priority organism for the development of new antibiotics. The species accumulates **antimicrobial resistance (AMR) genes** through horizontal gene transfer mediated by diverse mobile genetic elements, including plasmids and genomic islands. Chromosomal **resistance islands**, which integrate at recurrent insertion hotspots, represent major reservoirs of acquired resistance determinants in successful multidrug-resistant lineages.

This tutorial demonstrates how [**PanGBank**](https://pangbank.genoscope.cns.fr) and [**PPanGGOLiN**](https://github.com/labgem/PPanGGOLIN) can be used to characterize the distribution of AMR genes at the pangenome scale and how a newly sequenced isolate can be integrated into an existing pangenome context without the need to recompute the pangenome from scratch.

**Part 1 — AMR Gene Distribution in the Pangenome**  
A pre-computed *A. baumannii* pangenome is retrieved from the PanGBank GTDB_refseq collection. Its gene families are annotated with [AMRFinderPlus](https://github.com/ncbi/amr), and the distribution of AMR genes across the *persistent*, *shell*, and *cloud* genome partitions is examined. Insertion spots are then analysed to identify chromosomal loci most frequently associated with antimicrobial resistance.

**Part 2 — Projection of a New Genome**  
A canine *A. baumannii* isolate, the ABO21-A003 strain, is projected onto the existing pangenome. Each gene is assigned to a pangenome family, its chromosomal location is matched against the known insertion spot catalogue, and all Regions of Genomic Plasticity (RGPs) carrying AMR genes are identified.

> **Before starting**, run the first code cell below once to set up the environment.


In [ ]:
%%capture pangbank_tutorials_init_logs

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os
from shutil import which
from pathlib import Path

conda_command = ""
if IN_COLAB:
    !pip install pyvis networkx pygenomeviz pangbank-api[sdk]==0.5.0 fa2 kaleido
    !apt install git-lfs
    !git clone https://github.com/labgem/PanGBank-tutorial.git
    !ln -s PanGBank-tutorial/tutorials/article_use_case/ppanggolin_genome_output ppanggolin_genome_output
    !ln -s PanGBank-tutorial/tutorials/article_use_case/amrfinder_result.tsv amrfinder_result.tsv
    !ln -s PanGBank-tutorial/tutorials/article_use_case/ppanggolin_output ppanggolin_output
    !ln -s PanGBank-tutorial/tutorials/article_use_case/projection_output projection_output
    !ln -s PanGBank-tutorial/tutorials/article_use_case/rgp_cluster rgp_cluster
!curl -o datasets 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets'
!curl -o dataformat 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/dataformat'
!chmod +x datasets dataformat

---

## Overview: Precomputed Data

The command-line steps described in each part can be **computationally intensive** and may not always be suitable for execution in a notebook environment.

> **All outputs have been precomputed and are already provided in this repository.** You do not need to run any shell commands to follow the tutorial. The step descriptions are included for transparency and reproducibility.

| File / Directory | Produced by | Description |
|---|---|---|
| `amrfinder_result.tsv` | Part 1 – Step 3 | AMRFinderPlus annotations for all pangenome gene families |
| `ppanggolin_output/` | Part 1 – Step 5 | Pangenome flat-file exports (partitions, RGPs, spots and modules) |
| `ppanggolin_genome_output/` | Part 1 – AbaR1 section | Per-genome GFF files with pangenome annotations, used for the AbaR1 visualization |
| `projection_output/` | Part 2 – Step 7 | All projection outputs for the canine isolate |

---

## Step 1: Download Pangenome from PanGBank

Search for and download the *A. baumannii* pangenome from the `GTDB_refseq` collection using [PanGBank Command-Line Interface](https://github.com/labgem/PanGBank-cli). The `--release-version` option pins the download to a specific data release to ensure reproducibility.

#### Commands

```bash
pangbank search-pangenomes \
    --collection GTDB_refseq \
    --taxon "s__Acinetobacter baumannii" \
    --download \
    --release-version 2.0.0
```

| | File | Description |
|---|---|---|
| **Input** | — | Queries the PanGBank API (no local file required) |
| **Output** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | Pangenome in HDF5 format |
---

Once downloaded, inspect the pangenome content with:

```bash
ppanggolin info \
    -p ./pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5 \
    --content
```

| | File | Description |
|---|---|---|
| **Input** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | Downloaded pangenome HDF5 file |
| **Output** | — | Prints pangenome statistics to stdout, including genome count, gene family count, partition sizes, RGPs, spots and modules |

## Step 2: Extract Gene Family Sequences

Export one representative protein sequence per gene family from the pangenome. This FASTA file is used as input for AMRFinderPlus in the next step. Running AMRFinderPlus at the gene family level rather than on individual genomes avoids repeated annotation of identical or highly similar genes across thousands of genomes while retaining a direct link between AMR determinants and pangenome families.

#### Command

```bash
ppanggolin fasta \
    -p ./pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5 \
    --prot_families all \
    --compress \
    -f \
    -o families_faa_output
```

| Flag | Meaning |
|---|---|
| `--prot_families all` | Export representative sequences for all gene families, regardless of partition |
| `--compress` | Compress the output FASTA file with gzip |
| `-f` | Overwrite the output directory if it already exists |
| `-o families_faa_output` | Output directory |

| | File | Description |
|---|---|---|
| **Input** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | Pangenome HDF5 file |
| **Output** | `families_faa_output/all_protein_families.faa.gz` | Gzip-compressed FASTA file containing one representative protein sequence per gene family, with the family ID used as the sequence header |

## Step 3: Annotate AMR Genes with AMRFinderPlus

Run [AMRFinderPlus](https://github.com/ncbi/amr) on the representative protein sequences of pangenome gene families AMR, virulence, and stress response genes. Because the input sequences represent gene families rather than individual genomes, each annotation directly maps to a pangenome family.

#### Command

```bash
amrfinder \
    -p families_faa_output/all_protein_families.faa.gz \
    --plus \
    --threads 8 \
    -o amrfinder_result.tsv
```

| Flag | Meaning |
|---|---|
| `-p` | Protein FASTA file input |
| `--plus` | Also report virulence factors and stress response genes |
| `--threads 8` | Number of parallel threads |
| `-o` | Output file |

| | File | Description |
|---|---|---|
| **Input** | `families_faa_output/all_protein_families.faa.gz` | Gzio FASTA file of representative protein sequences for all gene families |
| **Output** | `amrfinder_result.tsv` | Tab-separated table with one row per annotated gene family. Columns include element symbol, element name, AMR class, subclass, detection method, and alignment statistics |

## Step 4: Embed AMRFinder Annotations into the Pangenome

Embed the AMRFinderPlus results directly into the pangenome HDF5 file so that each gene family carries its associated AMR metadata. Once embedded, downstream PPanGGOLiN commands (e.g. `ppanggolin write_genomes` and `ppanggolin projection`) automatically propagate these annotations to individual genome outputs.

The AMRFinderPlus TSV requires minor header normalisation before import:
- `%` characters are replaced with `Prct` to avoid column-name parsing issues
- spaces in column names are replaced with underscores
- the `Protein id` column is renamed to `families` to match the identifier expected by PPanGGOLiN

#### Commands

```bash
# Normalise the AMRFinder TSV header in-place
sed -i 's/\%/Prct/g' amrfinder_result.tsv
sed -i '1s/ /_/g' amrfinder_result.tsv
sed -i '1s/\bProtein_id\b/families/' amrfinder_result.tsv

# Embed the annotations into the pangenome
ppanggolin metadata \
    -p ./pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5 \
    --metadata amrfinder_result.tsv \
    --source amrfinder \
    --assign families
```

| Flag | Meaning |
|---|---|
| `--metadata` | Annotation TSV file to import; must contain a column matching the `--assign` target |
| `--source amrfinder` | Label assigned to this metadata block within the HDF5 file |
| `--assign families` | Attach metadata rows to gene families by matching their family identifiers |

| | File | Description |
|---|---|---|
| **Input** | `amrfinder_result.tsv` | AMRFinderPlus annotation table after header normalisation |
| **Input** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | Pangenome HDF5 file |
| **Output** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | Updated HDF5 file (modified in place); gene families now carry `amrfinder` metadata annotations |

## Step 5: Export Pangenome Data

Write flat-file outputs from the pangenome for downstream analysis. Each flag activates a different output type.

#### Command

```bash
ppanggolin write_pangenome \
    -p ./pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5 \
    --spots \
    --regions \
    --families_tsv \
    --partitions \
    --regions_families \
    --modules \
    --spot_modules \
    -o ppanggolin_output \
    -f
```

| Flag | Meaning |
|---|---|
| `--spots` | Export spot assignments for each RGP |
| `--regions` | Export RGP coordinates per genome |
| `--families_tsv` | Export the gene family membership table |
| `--partitions` | Export one text file per partition (*persistent*, *shell*, *cloud*) listing family IDs |
| `--regions_families` | Export the mapping between RGPs and their constituent gene families |
| `--modules` | Export functional module composition |
| `--spot_modules` | Export which modules are found in which spots |
| `-f` | Overwrite the output directory if it already exists |

| | File | Description |
|---|---|---|
| **Input** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | The pangenome HDF5 file (with embedded AMRFinder metadata from Step 4) |
| **Output** | `ppanggolin_output/gene_families.tsv` | Gene family membership — maps each gene (per genome) to its family ID |
| **Output** | `ppanggolin_output/regions_of_genomic_plasticity.tsv` | RGP coordinates and genome assignments |
| **Output** | `ppanggolin_output/rgp_families.tsv` | Gene families belonging to each RGP |
| **Output** | `ppanggolin_output/spots.tsv` | RGP-to-spot assignments |
| **Output** | `ppanggolin_output/partitions/persistent.txt` | Family IDs assigned to the *Persistent* partition |
| **Output** | `ppanggolin_output/partitions/shell.txt` | Family IDs assigned to the *Shell* partition |
| **Output** | `ppanggolin_output/partitions/cloud.txt` | Family IDs assigned to the *Cloud* partition |
| **Output** | `ppanggolin_output/functional_modules.tsv` | Gene family to functional module assignments |
| **Output** | `ppanggolin_output/modules_spots.tsv` | Association between functional modules and spots |
| **Output** | `ppanggolin_output/summarize_spots.tsv` | Per-spot summary statistics (RGP count, genome count, etc.) |

---

# Part 1 — AMR Gene Distribution in the Pangenome

## Biological Context

The resistance phenotype of *A. baumannii* results from the combination of intrinsic and acquired resistance mechanisms. **Intrinsic resistance** determinants encoded in the conserved genome (*persistent* partition) provide a baseline level of tolerance to multiple antibiotic classes. In addition, strains continuously acquire **mobile resistance genes** through horizontal gene transfer, with many resistance islands integrating at specific chromosomal loci.

PPanGGOLiN partitions gene families into three groups based on their occurrence across the genomes:

| Partition | Presence across genomes |
|---|---|
| ***Persistent*** | Near-universal |
| ***Shell*** | Intermediate frequency |
| ***Cloud*** | Rare or strain-specific presence |

Overlaying AMRFinderPlus annotations onto these partitions and onto the [spots of insertion](https://ppanggolin.readthedocs.io/en/latest/user/RGP/rgpAnalyses.html#spot-prediction) enables the characterization of which resistance genes are intrinsic versus potentially mobilizable, which AMR classes dominate the variable resistome, and which insertion loci are most strongly associated with resistance gene acquisition.

In [ ]:
import pandas as pd
from pathlib import Path
import plotly.express as px


In [ ]:
resistance_annotation_file = "amrfinder_result.tsv"
df_amrfinder = pd.read_csv(resistance_annotation_file, sep="\t")

df_amrfinder["amrfinder_annotation"] = True
df_amrfinder

## Filter AMR Results

AMRFinderPlus was run with `--plus` option, which extends the standard AMR database to include virulence factors, stress response genes, and other resistance-associated elements. The full output comprises 164 annotated families. Entries are restricted to those where `Type == "AMR"`, excluding the 41 hits classified as `STRESS` or `VIRULENCE`. The resulting set of **123 AMR gene families** is used for all downstream analyses.

In [ ]:
arm_filter = df_amrfinder['Type'] == "AMR"
df_amr = df_amrfinder.loc[arm_filter][['Protein id', 'Element symbol', 'Element name', 'Scope', 'Type',
       'Subtype', 'Class', 'Subclass', 'Method',
       'HMM description', 'amrfinder_annotation']]

df_amr

## Assign Pangenome Partitions to AMR Gene Families

Each gene family belongs to exactly one partition. The partition lists exported in Step 5 are used to assign a partition label (*persistent*, *shell*, or *cloud*) to each of the 123 AMR families.

This classification is central to distinguishing genes encoded in the conserved core genome from variable genes found only in subsets of strains, including genes that may be associated with mobile genetic elements.

In [ ]:
def parse_partition_file(p_file):
    with open(p_file) as fl:
        return [l.strip() for l in fl]

shell_fams = parse_partition_file('ppanggolin_output/partitions/shell.txt')
cloud_fams = parse_partition_file('ppanggolin_output/partitions/cloud.txt')
persistent_fams = parse_partition_file('ppanggolin_output/partitions/persistent.txt')

# 'Protein id' is the family representative sequence ID — the same ID used as the FASTA header
# fed to AMRFinder, so it is the natural key linking AMRFinder hits back to pangenome families.
df_amr.loc[df_amr['Protein id'].isin(persistent_fams), 'partition'] = "Persistent"
df_amr.loc[df_amr['Protein id'].isin(cloud_fams), 'partition'] = "Cloud"
df_amr.loc[df_amr['Protein id'].isin(shell_fams), 'partition'] = "Shell"

df_amr

### AMR Gene Family Distribution by Partition

The two charts explore the distribution of the 123 AMR gene families across pangenome partitions.

The **first chart** reports the total number of AMR families in each partition, whereas the **second chart** stratifies these families by AMR class.

In [ ]:
partition_to_color = {'Persistent': '#e59c04', 'Shell': '#00d860', 'Cloud': '#79deff'}
partition_order = ["Persistent", "Shell", "Cloud"]

df_amr_rgp_partition_total = df_amr.groupby(["partition"]).agg({
                                    "Protein id":"count"}).reset_index()

fig = px.bar(df_amr_rgp_partition_total, x='partition', y='Protein id', color="partition",
             color_discrete_map=partition_to_color,
             category_orders={"partition": partition_order},
             title="AMR Gene Families per Partition",
             text='Protein id',
             labels={"Protein id": "# Gene Families<br>with AMR annotation", "partition": "Pangenome Partition"})

fig.update_traces(textfont_size=12, width=0.6)

fig.update_layout(
    template="simple_white",
    font=dict(
        family="Arial",
        size=14,
    ),
    autosize=False,
    width=700,
    height=450,
)
fig.show()
#fig.write_image("figureS3.png")
#fig.write_image("figureS3.svg")

In [ ]:
df_amr_rgp_partition = df_amr.groupby(["partition", "Class"]).agg({
                                    "Protein id":"count"}).reset_index()

fig = px.bar(df_amr_rgp_partition, x='Class', y='Protein id', color="partition",
             color_discrete_map=partition_to_color,
             category_orders={"partition": partition_order},
             title="AMR Gene Families by Class and Partition",
             labels={"Protein id": "# Gene Families", "Class": "AMR Class", "partition": "Pangenome Partition"})

fig.update_layout(
            template="simple_white",
            font=dict(
                family="Arial",
                size=12,
            ),
            #autosize=False,
            width=1000,
            height=700,
)
fig.update_xaxes(tickangle=45)
fig.show()

#fig.write_image("figureS4.png")
#fig.write_image("figureS4.svg")

In [ ]:
df_amr_persistent =df_amr.loc[df_amr['partition'] == "Persistent"][['Protein id', "partition", "Element symbol", "Element name", "Type", "Class"]]
df_amr_persistent

These five *persistent* AMR gene families correspond to well-characterised intrinsic resistance determinants conserved across *A. baumannii* strains.

| Gene | Class | Notes |
|---|---|---|
| `blaADC` | Beta-lactam | AmpC cephalosporinase |
| `blaOXA` (OXA-51 family) | Beta-lactam | Chromosomal OXA-type carbapenemase, a defining marker of the species |
| `amvA` | Efflux | Multidrug efflux pump contributing to intrinsic tolerance |
| `cxpE` | Phenicol | Chloramphenicol efflux transporter |
| `ant(3'')-IIa` | Aminoglycoside | Aminoglycoside-modifying enzyme |

Their *persistent* status confirms that these resistance determinants are conserved across essentially all strains in the analysed pangenome, independently of clinical context. In contrast, *shell* and *cloud* AMR families represent variable resistance determinants that are restricted to subsets of strains and may include genes associated with mobile genetic elements.

---

## Building the RGP–Spot–AMR Analysis Table

To analyse how AMR genes are distributed across RGP and insertion spots, we assemble a unified table by integrating four sources:

1. **RGP gene families** (`rgp_families.tsv`) — gene families present in each RGP  
2. **Spot assignments** (`spots.tsv`) — insertion spots associated with each RGP (RGPs not assigned to any spot are labelled `"No spot"`)  
3. **Functional modules** (`functional_modules.tsv`) — groups of co-occurring and co-localized gene families representing potential functional units
4. **AMR annotations** — the AMRFinderPlus annotation table containing the 123 AMR gene families identified above  

Each row of the merged table represents one gene family within one RGP and is enriched with its associated spot, functional module, and AMR annotation when available.

### Load Spot Assignments

In [ ]:
df_spots = pd.read_csv('ppanggolin_output/spots.tsv', sep='\t')
df_spots

### Gene Families per RGP

Each row links one gene family to the RGP it belongs to in a specific genome. This table drives both the spot-level statistics and the Jaccard comparisons in Part 2.

In [ ]:
# read rgps families
rgp_families_file = Path('ppanggolin_output/rgp_families.tsv')
df_rgp_fams = pd.read_csv(rgp_families_file, sep='\t')

df_rgp_fams

### Load Functional Modules

[Functional modules](https://ppanggolin.readthedocs.io/en/latest/user/Modules/moduleAnalyses.html) are groups of variable genome genes that frequently co-occur across pangenome genomes. They may correspond to functional units or mobile genetic elements that are frequently transferred together. Linking AMR families to modules can reveal whether resistance genes are consistently associated with specific sets of variable genes.

In [ ]:
module_families_file = Path("ppanggolin_output/functional_modules.tsv")
df_module_fams = pd.read_csv(module_families_file, sep="\t")

df_module_fams

### Merge All Data Tables

The four tables are joined using their shared keys, producing one row per (RGP, gene family) pair enriched with spot, module, and AMR annotations where available. The 123 AMR gene families represent only a small fraction of the total gene families in the pangenome; therefore, most rows will not contain an AMR annotation.

In [ ]:
# Left joins throughout: keep every RGP–family pair even when a family has no spot,
# module, or AMR annotation — unmatched rows get NaN, filled to "No spot" where needed.
df_rgp_info_merged = df_rgp_fams.merge(df_spots, on="rgp_id", how="left")
df_rgp_info_merged['spot_id'] = df_rgp_info_merged['spot_id'].fillna("No spot")
df_rgp_info_merged = df_rgp_info_merged.merge(df_module_fams, on="family_id", how="left")
df_rgp_info_merged = df_rgp_info_merged.merge(df_amr, left_on="family_id", right_on="Protein id", how="left")
df_rgp_info_merged

---

# RGP and Spot-Level Analysis

A [Region of Genomic Plasticity (RGP)](https://ppanggolin.readthedocs.io/en/latest/user/RGP/rgpAnalyses.html) is a genome-specific cluster of *shell* and *cloud* genes. Many RGPs originate from horizontal gene transfer (HGT) events and correspond to genomic islands (GIs).

[Spots of insertion](https://ppanggolin.readthedocs.io/en/latest/user/RGP/rgpAnalyses.html#spot-prediction) group RGPs from **different genomes** that share the same chromosomal insertion locus, identified through conserved **flanking** ***persistent*** **genes**. Importantly, spot membership is based on genomic location rather than gene content: two RGPs assigned to the same spot may contain different sets of accessory genes or mobile genetic elements. The number of RGPs assigned to a spot corresponds to the number of genomes in the pangenome containing an insertion at that locus.

Some **spots** accumulate hundreds of **RGPs** across many strains, indicating recurrent integration at permissive chromosomal loci, whereas others are observed in only one or a few genomes. Summarising AMR gene content per spot identifies chromosomal loci that are preferentially associated with the acquisition of resistance determinants and highlights insertion sites that may correspond to resistance islands.


## Summary Statistics per Spot

The following metrics are computed for each spot:
- **n_rgps**: number of genomes with an RGP at this locus
- **n_families / n_modules**: total gene-family and module diversity across all RGPs at this spot
- **n_rgps_with_amr**: number of RGPs at this spot that contain at least one AMR gene family
- **prct_rgp_with_amr**: proportion of RGPs at this spot that carry at least one AMR family (%)

In [ ]:
has_amr = df_rgp_info_merged['amrfinder_annotation'].notna()

# Count unique RGPs, families, and modules per spot across all rows
overall = df_rgp_info_merged.groupby('spot_id').agg(
    n_rgps=('rgp_id', 'nunique'),
    n_families=('family_id', 'nunique'),
    n_modules=('module_id', 'nunique')
)

# Same aggregation but restricted to AMR-annotated rows. Gives the number of RGPs
# at each spot that carry at least one AMR gene family
amr_only = df_rgp_info_merged.loc[has_amr].groupby('spot_id').agg(
    n_rgps_with_amr=('rgp_id', 'nunique'),
    n_families_with_amr=('family_id', 'nunique'),
    n_modules_with_amr=('module_id', 'nunique')
)

# Left join: spots with zero AMR RGPs are absent from amr_only, so fillna(0) is correct
spot_summary = overall.join(amr_only, how='left').fillna(0).astype(int).reset_index()

spot_summary['prct_rgp_with_amr'] = (spot_summary['n_rgps_with_amr'] / spot_summary['n_rgps'] * 100).round(2)
spot_summary

### Spot AMR Enrichment Scatter Plot

Only spots containing at least one AMR gene family are shown.

Each point represents a single spot:

- **X-axis**: number of RGPs assigned to the spot, corresponding to the number of genomes carrying an insertion at this locus
- **Y-axis**: percentage of RGPs within the spot that contain at least one AMR gene family
- **Point size**: mean number of AMR gene families per RGP within the spot
- **Colour**: number of distinct AMR gene families detected across all RGPs assigned to the spot


In [ ]:
# Exclude spots with no spot assignment and spots with no AMR content
plot_filter = (spot_summary["spot_id"] != "No spot") & (spot_summary["n_rgps_with_amr"] > 0)

rgp_size = df_rgp_fams.groupby('rgp_id')['family_id'].count().reset_index()
rgp_size.columns = ['rgp_id', 'n_families_in_rgp']

rgp_size_with_spot = rgp_size.merge(df_spots, on='rgp_id', how='left')
rgp_size_with_spot['spot_id'] = rgp_size_with_spot['spot_id'].fillna('No spot')

amr_rgp_size = (
    df_rgp_info_merged[df_rgp_info_merged["amrfinder_annotation"].notna()]
    .groupby(["spot_id", "rgp_id"])["family_id"]
    .nunique()
    .reset_index()
    .rename(columns={"family_id": "n_amr_fams"})
)

mean_amr_per_spot = (
    amr_rgp_size.groupby("spot_id")["n_amr_fams"]
    .mean()
    .reset_index()
    .rename(columns={"n_amr_fams": "mean_amr_families_in_rgp"})
)

spot_summary = spot_summary.drop(columns=["mean_amr_families_in_rgp"], errors="ignore")
spot_summary = spot_summary.merge(mean_amr_per_spot, on="spot_id", how="left")

spot_summary["label"] = spot_summary["spot_id"].where(
    (spot_summary["n_rgps"] > 450) & (spot_summary["prct_rgp_with_amr"] > 15), ""
)
fig = px.scatter(
    spot_summary.loc[plot_filter],
    x="n_rgps",
    y="prct_rgp_with_amr",
    hover_data=spot_summary.columns,
    color="n_families_with_amr",
    size="mean_amr_families_in_rgp",
    text="label",
    title="AMR Gene Family Prevalence Across Genomic Spots",
)
fig.update_coloraxes(colorscale="Viridis")
fig.update_traces(textposition="top center")
fig.update_layout(
    autosize=False,
    width=1000,
    height=600,
    xaxis_title="# Genomes",
    yaxis_title="% RGPs with AMR gene families",
    coloraxis_colorbar=dict(title="# AMR families"),
    template="simple_white",
    font=dict(
        family="Arial",
        size=14,
    ),
)

fig.show()
#fig.write_image("figureS5.png")
#fig.write_image("figureS5.svg")

### Identifying Key AMR Hotspots

From the scatter plot, three spots stand out as **major AMR hotspots**, each present in more than 500 genomes and with more than 40% of their associated RGPs carrying AMR genes:

- **spot_47** — the most diverse AMR hotspot, corresponding to the chromosomal insertion locus of the AbaR1 resistance island
- **spot_7** and **spot_11** — two other recurrent integration loci strongly associated with resistance gene acquisition

To determine which resistance islands are associated with the **spot_47** insertion locus, we inspect the genomes carrying RGPs at this locus and examine their AMR gene content.

In [ ]:
# Inspect spot_47: which genomes carry RGPs here?
focus_spot = "spot_47"

df_pan_rgps_detail = pd.read_csv("ppanggolin_output/regions_of_genomic_plasticity.tsv", sep="\t")
spot_rgp_ids = df_spots[df_spots["spot_id"] == focus_spot]["rgp_id"]
spot_rgp_detail = (df_pan_rgps_detail[df_pan_rgps_detail["region"].isin(spot_rgp_ids)]
                   [["region", "genome", "genes", "length"]]
                   .sort_values("length", ascending=False)
                   .reset_index(drop=True))

row = spot_summary.loc[spot_summary["spot_id"] == focus_spot].iloc[0]
print(f"{focus_spot}: {int(row['n_rgps_with_amr'])}/{int(row['n_rgps'])} RGPs carry AMR "
      f"({row['prct_rgp_with_amr']:.0f}%),  {int(row['n_families_with_amr'])} distinct AMR families")
print()
display(spot_rgp_detail.head(10))

One of the genomes containing an RGP at **spot_47** is **GCF_000069245.1** — the assembly accession for *A. baumannii* strain **AYE** ([NCBI](https://www.ncbi.nlm.nih.gov/assembly/GCF_000069245.1/)), in which [Fournier *et al.* (2006, *PLoS Genetics*)](https://pmc.ncbi.nlm.nih.gov/articles/PMC1326220/) first characterised **AbaR1**. By extracting the AMR gene families from the GCF_000069245.1-specific RGP at **spot_47**, we can directly assess whether its gene content matches the published description of AbaR1.


In [ ]:
# Zoom in on the GCF_000069245.1 (strain AYE) RGP specifically
aye_genome = "GCF_000069245.1"
aye_row = spot_rgp_detail[spot_rgp_detail["genome"] == aye_genome]
if not aye_row.empty:
    aye_rgp = aye_row.iloc[0]["region"]
    print(f"\nAMR gene families in {aye_genome} RGP ({aye_rgp}, "
          f"{aye_row.iloc[0]['length']:,} bp, {aye_row.iloc[0]['genes']} genes):")
    aye_amr = (df_rgp_info_merged
               .query(f"rgp_id == '{aye_rgp}'")
               .dropna(subset=["amrfinder_annotation"])
               [["Element symbol", "Element name", "Class", "Subclass", "spot_id"]]
               .drop_duplicates()
               .sort_values(["Class", "Element symbol"])
               .reset_index(drop=True))
    display(aye_amr)

---

## Genome-Level Visualisation: The AbaR1 Resistance Island

GCF_000069245.1 (strain AYE) has an ~86 kb RGP at spot_47 encoding carbapenemases, aminoglycoside-modifying enzymes, and sulfonamide resistance genes, and other resistance markers. This matches the description of **AbaR1** (*Acinetobacter baumannii* Resistance Island 1) in [Fournier *et al.* (2006, *PLoS Genetics*)](https://pmc.ncbi.nlm.nih.gov/articles/PMC1326220/), confirming that spot_47 is the AbaR1 locus.

The circular map below is generated with [CGView](https://cgview.ca/) from the Proksee JSON exported by `ppanggolin write_genomes`. AMRFinder metadata embedded in the pangenome is propagated to each CDS in the output, so AMR genes are highlighted directly from the pangenome annotations.

```bash
ppanggolin write_genomes \
    -p ./pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5 \
    --genomes GCF_000069245.1 \
    --gff --proksee \
    --add_metadata \
    -o ppanggolin_genome_output
```

| | File | Description |
|---|---|---|
| **Input** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | Pangenome with embedded AMRFinder metadata |
| **Output** | `ppanggolin_genome_output/gff/GCF_000069245.1.gff` | GFF with partition, family, RGP, spot, and AMRFinder attributes per CDS |
| **Output** | `ppanggolin_genome_output/proksee/GCF_000069245.1.json` | CGView JSON for circular map rendering |


In [ ]:
import json
from IPython.display import display, HTML

with open("ppanggolin_genome_output/proksee/GCF_000069245.1.json") as f:
    data = json.load(f)

# Add a legend entry for the AMR track
data['cgview']['legend']['items'].append({
    'decoration': 'arrow',
    'name': 'AMR',
    'swatchColor': '#e74c3c'
})

# Add a dedicated AMR track as a second outside ring
data['cgview']['tracks'].append({
    'dataKeys': 'AMR',
    'dataMethod': 'source',
    'dataType': 'feature',
    'name': 'AMR',
    'position': 'outside',
    'separateFeaturesBy': 'strand',
    'thicknessRatio': 0.5
})

# Duplicate each AMR-annotated gene as a new feature on the AMR track
for feature in list(data['cgview']['features']):
    meta = feature.get('meta', {})
    if any(k.startswith('family_amrfinder_') for k in meta):
        symbol = meta.get('family_amrfinder_Element_symbol', ['AMR'])
        name = symbol[0] if isinstance(symbol, list) else symbol
        data['cgview']['features'].append({
            'contig':  feature['contig'],
            'legend':  'AMR',
            'meta':    meta,
            'name':    name,
            'source':  'AMR',
            'start':   feature['start'],
            'stop':    feature['stop'],
            'strand':  feature.get('strand', 1),
            'type':    'AMR'
        })

# RGP loci on NC_010410.1 (first contig, mapStart=0)
rgp_loci = {
    'spot_47 (AbaR1)': (3610132, 3696120),
    'spot_7':          (2341346, 2485362),
    'spot_11':         (1790570, 1934871),
}


out_path = "GCF_000069245.1_amr.json"
with open(out_path, "w") as _f:
    json.dump(data, _f)
print(f"Saved {out_path}")
json_str  = json.dumps(data)

btn_style = ("margin:4px 6px;padding:5px 12px;border-radius:4px;"
             "border:1px solid #888;cursor:pointer;font-size:13px;")

def focus_js(start, stop):
    pad = (stop - start) // 2
    total = "window.cgviewAYE.sequence.length"
    return (
        f"if(window.cgviewAYE){{"
        f"var c=window.cgviewAYE.contigs('NC_010410.1');"
        f"var off=c?c.mapStart:0;"
        f"window.cgviewAYE.moveTo(1,{total},{{duration:0,callback:function(){{"
        f"window.cgviewAYE.moveTo(off+{start}-{pad},off+{stop}+{pad},{{duration:600}});"
        f"}}}})}}"
    )

buttons_html = " ".join(
    f'<button style="{btn_style}" onclick="{focus_js(s, e)}">{label}</button>'
    for label, (s, e) in rgp_loci.items()
)
buttons_html += (
    f' <button style="{btn_style}background:#f0f0f0;" '
    f'onclick="if(window.cgviewAYE)window.cgviewAYE.reset()">Reset</button>'
)

html = f"""
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/cgview/dist/cgview.css">
<div id="cgview-aye" style="width:800px;height:600px;border:1px solid #ccc;">Loading CGView...</div>
<div style="margin-top:6px;">{buttons_html}</div>
<script>
(function() {{
    function loadScript(src) {{
        return new Promise(function(resolve, reject) {{
            var s = document.createElement('script');
            s.src = src; s.onload = resolve; s.onerror = reject;
            document.head.appendChild(s);
        }});
    }}
    Promise.resolve()
        .then(function() {{ return loadScript('https://cdn.jsdelivr.net/npm/d3@7'); }})
        .then(function() {{ return loadScript('https://cdn.jsdelivr.net/npm/cgview/dist/cgview.min.js'); }})
        .then(function() {{
            var jsonData = {json_str};
            document.getElementById('cgview-aye').innerHTML = '';
            var viewer = new CGView.Viewer('cgview-aye', {{width: 800, height: 600}});
            viewer.io.loadJSON(jsonData);
            window.cgviewAYE = viewer;
        }})
        .catch(function(err) {{
            document.getElementById('cgview-aye').innerText = 'Failed to load CGView: ' + err;
        }});
}})();
</script>
"""

display(HTML(html))

### AMR Gene Content per RGP in Strain AYE

The table below summarises the number of AMR-annotated CDS in each RGP of strain AYE (GCF_000069245.1), sorted by descending gene count.

In [ ]:
# AYE RGP IDs from the pangenome regions table
aye_rgp_ids = df_pan_rgps_detail[df_pan_rgps_detail["genome"] == "GCF_000069245.1"]["region"]

# AMR families in AYE RGPs, reuses df_rgp_info_merged already built above
aye_amr = df_rgp_info_merged[
    df_rgp_info_merged["rgp_id"].isin(aye_rgp_ids) &
    df_rgp_info_merged["amrfinder_annotation"].notna()
]

df_aye_rgp_amr = (
    aye_amr.groupby(["rgp_id", "spot_id"])
    .agg(**{
        "nb AMR families": ("Element symbol", "count"),
        "Drug classes":    ("Class",          lambda x: ", ".join(sorted(x.dropna().unique()))),
        "AMR genes":       ("Element symbol", lambda x: ", ".join(sorted(x.dropna().unique()))),
    })
    .sort_values("nb AMR families", ascending=False)
    .reset_index()
)

display(df_aye_rgp_amr)


---

## RGP Clustering

RGPs across all genomes can be grouped into clusters of structurally similar islands using `ppanggolin rgp_cluster`. Two RGPs are placed in the same cluster when their Gene Repertoire Relatedness (GRR) score exceeds a similarity threshold. The GRR is computed here as the number of gene families shared between two RGPs divided by the larger of their two family counts (*max_grr*). RGPs with a max_grr above **0.8** are clustered together, capturing groups of genomic islands that share at least 80% of their gene family content.

```bash
ppanggolin rgp_cluster \
    -p ./pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5 \
    --grr_metric max_grr \
    --min_grr 0.8 \
    -o rgp_cluster -f \
    --add_metadata
```

| | File | Description |
|---|---|---|
| **Input** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | Pangenome HDF5 |
| **Output** | `rgp_cluster/rgp_cluster.tsv` | RGP → cluster assignment with spot annotation |
| **Output** | `rgp_cluster/rgp_cluster.graphml` | Similarity graph (nodes = RGPs, edges = GRR scores) |

> **All outputs have been precomputed and are available in `rgp_cluster/`.**


In [ ]:
df_rgp_clusters = pd.read_csv("rgp_cluster/rgp_cluster.tsv", sep="\t")
display(df_rgp_clusters)

### Genetic Diversity at spot_47

Focusing on spot_47, each RGP cluster represents a distinct genetic configuration of the AbaR1 resistance island.  A [previous study](https://doi.org/10.1038/s41467-022-32829-5) reported 53 configurations across *A. baumannii*; the pangenome here reveals **56** clusters.


In [ ]:
print(f"{df_rgp_clusters['cluster'].nunique():,} clusters across {len(df_rgp_clusters):,} RGPs")
print(f"{len(df_rgp_clusters[df_rgp_clusters['spot_id'] == 'spot_47']['cluster'].unique())} clusters in spot_47 ({len(df_rgp_clusters[df_rgp_clusters['spot_id'] == 'spot_47'])} RGPs)")

The bar chart summarizes the **spot_47** cluster composition. Each bar represents a distinct gene-content configuration of the AbaR1 resistance island locus, and its height corresponds to the number of genomes assigned to that cluster.

Clusters containing at least one AMR gene family are highlighted in red, enabling rapid identification of configurations associated with resistance. This plot can be used to compare cluster prevalence and prioritize specific configurations for further inspection.

In [ ]:
import networkx as nx
import plotly.express as px

# Load graph
graph = nx.read_graphml("rgp_cluster/rgp_cluster.graphml")

# Clusters present at spot_47
spot47_clusters = df_rgp_clusters[df_rgp_clusters["spot_id"] == "spot_47"]["cluster"].unique()
print(f"{len(spot47_clusters)} distinct genetic configurations at spot_47")

# AMR-carrying RGPs at spot_47 (from the merged table built earlier)
amr_rgp_ids = set(
    df_rgp_info_merged[df_rgp_info_merged["amrfinder_annotation"].notna()]["rgp_id"]
)
amr_clusters = set(
    df_rgp_clusters[df_rgp_clusters["RGPs"].isin(amr_rgp_ids)]["cluster"]
)

# Per-cluster genome count: sum identical_rgp_count (each node = group of identical RGPs)
rows = []
for n, d in graph.nodes(data=True):
    c = d.get("max_grr_cluster")
    if c not in spot47_clusters:
        continue
    rows.append({
        "cluster":   c,
        "n_genomes": d.get("identical_rgp_count", 1),
        "has_amr":   c in amr_clusters,
    })

df_spot47 = (
    pd.DataFrame(rows)
    .groupby(["cluster", "has_amr"], as_index=False)["n_genomes"].sum()
    .sort_values("n_genomes", ascending=False)
    .reset_index(drop=True)
)
df_spot47["color"] = df_spot47["has_amr"].map({True: "AMR", False: "No AMR"})

fig = px.bar(
    df_spot47, x="cluster", y="n_genomes", color="color",
    color_discrete_map={"AMR": "#e74c3c", "No AMR": "#aec6cf"},
    labels={"cluster": "Cluster", "n_genomes": "Number of genomes", "color": ""},
    title=f"spot_47 - {len(spot47_clusters)} genetic configurations",
)
fig.update_layout(xaxis_tickangle=45, xaxis_tickfont_size=9, showlegend=True)
fig.show()

In [ ]:
n_amr = len(df_spot47[df_spot47["color"] == "AMR"])
print(f"{n_amr} of the {len(df_spot47)} spot_47 clusters contain at least one AMR gene family")

RGPs associated with `spot_47` are extracted from the RGP cluster graph (`rgp_cluster.graphml`).

A ForceAtlas2 layout is computed on the graph to obtain a stable spatial representation, which is then used to generate an interactive network visualization with PyVis.

In this first version of the network, saved as `spot_47_network_by_amr.html`:
- Node size is proportional to the number of genomes represented by the identical-RGP group.
- Nodes carrying at least one AMR gene family are highlighted as red triangles.

In [ ]:
from pyvis.network import Network
from fa2 import ForceAtlas2

spot47_nodes = [n for n, d in graph.nodes(data=True) if d.get("spot_id") == "spot_47"]
sub = graph.subgraph(spot47_nodes).copy()

fa2 = ForceAtlas2(
    strongGravityMode=False,
    gravity=100,
    scalingRatio=1.0,
    verbose=False,
)
positions = fa2.forceatlas2_networkx_layout(sub, pos=None, iterations=500)
scale = 60

net = Network(notebook=True, cdn_resources="remote", height="600px", width="100%")
net.toggle_physics(False)
net.set_options('{"nodes": {"font": {"multi": "html"}}, "physics": {"enabled": false}}')
for n, d in sub.nodes(data=True):
    cluster   = d.get("max_grr_cluster", "")
    rgp_name  = d.get("name", "")
    rgp_list  = d.get("identical_rgp_names", "") or ""
    n_genomes = d.get("identical_rgp_count", 1)
    genome    = d.get("genome", "")
    n_fams    = d.get("families_count", "")
    has_amr   = cluster in amr_clusters

    x, y      = positions[n]
    title = (
        f"{rgp_name}\n"
        f"Cluster: {cluster}\n"
        f"Genome: {genome}\n"
        f"Identical RGPs: {n_genomes}\n"
        f"Families: {n_fams}\n"
        + (f"RGPs: {rgp_list[:120]}..." if len(rgp_list) > 120 else f"RGPs: {rgp_list}")
        + ("\n⚠ AMR" if has_amr else "")
    )
    net.add_node(
        n,
        label = f"{rgp_name} ({genome})",
        title = title,
        #group = cluster, # uncomment to color by cluster
        color = "#e74c3c" if has_amr else "#DDDDDD",
        size  = 30 + n_genomes ** 0.8 * 3,
        shape = "triangle" if has_amr else "dot",
        x     = float(x * scale),
        y     = float(y * scale),
    )

for u, v, d in sub.edges(data=True):
    net.add_edge(u, v, width=0.01, #value=d.get("max_grr", 1),
                 title=f"max_grr={d.get('max_grr', 0):.2f}")

net.save_graph("spot_47_network_by_amr.html")


The same network layout is reused to generate a second visualization where nodes are colored according to their RGP cluster membership.

The resulting interactive network is saved as `spot_47_network_by_cluster.html`.

In [ ]:
from itertools import cycle

palette = (
    px.colors.qualitative.Alphabet +
    px.colors.qualitative.Dark24 +
    px.colors.qualitative.Light24
)
cluster_colors = {
    c: color
    for c, color in zip(
        set(nx.get_node_attributes(sub, "max_grr_cluster").values()),
        cycle(palette),
    )
}

for node in net.nodes:
    cluster = sub.nodes[node["id"]]["max_grr_cluster"]
    node["color"] = cluster_colors[cluster]

net.save_graph("spot_47_network_by_cluster.html")


### Side-by-side visualization of the network

The two views represent the same RGP network using different node annotations:

- **Left:** network colored according to AMR status (AMR vs non-AMR).
- **Right:** network colored according to RGP cluster membership.

Hover over nodes and edges to inspect RGP cluster IDs, associated genomes, and `max_grr` similarity values.

In [ ]:
from IPython.display import HTML
from pathlib import Path
import base64

def html_to_iframe(path, height=700):
    html = Path(path).read_text(encoding="utf-8")
    encoded = base64.b64encode(html.encode()).decode()

    return f"""
    <iframe
        src="data:text/html;base64,{encoded}"
        width="100%"
        height="{height}"
        frameborder="0">
    </iframe>
    """

HTML(
    f"""
    <div style="display: flex; gap: 10px; width: 100%;">
        <div style="width: 50%;">
            {html_to_iframe("spot_47_network_by_amr.html")}
        </div>
        <div style="width: 50%;">
            {html_to_iframe("spot_47_network_by_cluster.html")}
        </div>
    </div>
    """
)

---

# Part 2 — Projection of a New Genome onto the Pangenome

## Biological Context

The **projection** feature of PPanGGOLiN enables the annotation of a new, external genome using an existing pangenome **without recomputing it**. Each gene in the incoming genome is assigned to the most similar pangenome gene family (or flagged as novel if no match is found), its Regions of Genomic Plasticity are detected, and those RGPs are matched to known spots based on their flanking *persistent* genes.

Projection is particularly suited to clinical and surveillance contexts: a single command yields a complete characterisation of a newly sequenced isolate — partition profile, completeness relative to the pangenome core, RGPs, spot assignments, and AMR gene content.

Here, *A. baumannii* strain **ABO21-A003** ([GCF_033191195.1](https://www.ncbi.nlm.nih.gov/datasets/genome/GCF_033191195.1/)), isolated from a dog during surveillance of a veterinary intensive care unit ([André *et al.* 2024](https://doi.org/10.3389/fvets.2024.1403376)), is projected onto the *A. baumannii* pangenome constructed in Part 1. The genome was not included in the PanGBank GTDB\_refseq collection. The key question is which AMR gene families this animal isolate carries and at which chromosomal loci they reside.

## Step 6: Download the New Genome

```bash
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/033/191/195/GCF_033191195.1_ASM3319119v1/GCF_033191195.1_ASM3319119v1_genomic.gbff.gz
```

| | File | Description |
|---|---|---|
| **Input** | — | NCBI FTP (no local file required) |
| **Output** | `GCF_033191195.1_ASM3319119v1_genomic.gbff.gz` | Annotated genome in GenBank flat-file format (gzip-compressed) |

## Step 7: Run the Projection

```bash
ppanggolin projection \
    -p ./pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5 \
    --anno GCF_033191195.1_ASM3319119v1_genomic.gbff.gz \
    --gff \
    --proksee \
    -o projection_output
```

| Flag | Meaning |
|---|---|
| `--anno` | Input genome annotation file (GenBank or GFF format) |
| `--gff` | Export an annotated GFF with partition, RGP, spot, and metadata attributes per gene |
| `--proksee` | Export a Proksee-compatible JSON for circular genome visualisation |
| `-o` | Output directory |

| | File | Description |
|---|---|---|
| **Input** | `pangbank/GTDB_refseq_s__Acinetobacter_baumannii_id10832.h5` | The existing pangenome (from Part 1) |
| **Input** | `GCF_033191195.1_ASM3319119v1_genomic.gbff.gz` | The new genome to project |
| **Output** | `projection_output/summary_projection.tsv` | Overall statistics: partition counts, RGP/spot/module counts, completeness |
| **Output** | `projection_output/input_genome/gene_to_gene_family.tsv` | Mapping from each gene in the new genome to its pangenome family |
| **Output** | `projection_output/input_genome/regions_of_genomic_plasticity.tsv` | RGPs detected in the new genome with coordinates |
| **Output** | `projection_output/input_genome/input_genome_rgp_to_spot.tsv` | RGP-to-spot assignments (matched by flanking *persistent* genes) |
| **Output** | `projection_output/input_genome/input_genome_proksee.json` | Proksee JSON for circular visualisation |

> **All outputs have been precomputed and are available in `projection_output/`.**


---

## Projection Results

The projection summary is first examined to assess how well ABO21-A003 fits within the *A. baumannii* pangenome — a high completeness score indicates that the isolate is a true *A. baumannii* and that most of its genes can be assigned to known pangenome families.


In [ ]:
import pandas as pd
import yaml

with open("projection_output/input_genome/projection_summary.yaml") as f:
    summary = yaml.safe_load(f)

# The YAML first key is literally "Projection_summary:Genome_name" (colon in the key name)
genome_name = summary["Projection_summary:Genome_name"]
print(f"Genome: GCF_033191195.1 (dog isolate)")
print(f"Completeness:  {summary['Completeness']}%")
print(f"Genes:         {summary['Genes']}")
print(f"Persistent:    {summary['Persistent']['genes']} genes / {summary['Persistent']['families']} families")
print(f"Shell:         {summary['Shell']['genes']} genes / {summary['Shell']['families']} families")
print(f"Cloud:         {summary['Cloud']['genes']} genes / {summary['Cloud']['families']} families")
print(f"  Specific:    {summary['Cloud']['specific families']} families (not in any pangenome family)")
print(f"RGPs detected: {summary['RGPs']}")
print(f"Spots matched: {summary['Spots']}")
print(f"Modules found: {summary['Modules']}")
print(f"New spots:     {summary['New_spots']}")

Strain ABO21-A003 integrates well into the *A. baumannii* pangenome, with **99.68% completeness** as computed by PPanGGOLiN — meaning nearly all *persistent* gene families are present. Completeness corresponds to the proportion of *persistent* gene families present in the genome (see the [PPanGGOLiN documentation on genome metrics](https://ppanggolin.readthedocs.io/en/latest/user/PangenomeAnalyses/pangenomeAnalyses.html)).

The RGP-to-spot assignment table below shows which of the detected RGPs are matched to known spots in the *A. baumannii* pangenome.


In [ ]:
df_proj_spots = pd.read_csv("projection_output/input_genome/input_genome_rgp_to_spot.tsv", sep="\t")
df_proj_rgps  = pd.read_csv("projection_output/input_genome/regions_of_genomic_plasticity.tsv", sep="\t")

df_proj = df_proj_rgps.merge(df_proj_spots, left_on="region", right_on="region", how="left")
df_proj["spot_id"] = df_proj["spot_id"].fillna("No spot")


print(f"\nAll {len(df_proj)} RGPs detected in the dog isolate:")
df_proj[["region", "length", "genes", "spot_id"]]

The canine isolate carries insertions at several known AMR hotspots. The AMR gene content table in the next section details all four RGPs that contain at least one AMR gene family:

- **CP136181.1_RGP_12** at **spot_47** — compact (~22 kb, 18 gene families) insertion at the AbaR1 locus; confirms that this hotspot is not restricted to nosocomial human strains.
- Two additional RGPs at **spot_7** and **spot_11** — two other major *A. baumannii* AMR hotspots.
- **CP136181.1_RGP_5** at **spot_31** — the largest AMR repertoire in this isolate. This RGP corresponds to the **ABGRI2** resistance island and carries three copies of *bla*<sub>TEM-1</sub> and two copies of *aph*A1, conferring resistance to beta-lactams and kanamycin respectively, consistent with the findings reported by André *et al.* (2024).

## Circular Genome Visualisation

The `--proksee` flag produces a JSON file compatible with [Proksee](https://proksee.ca/) and the [CGView.js](https://js.cgview.ca/) library. The circular map below renders the complete chromosome of ABO21-A003 with genes coloured by pangenome partition. A dedicated outer track in red highlights genes carrying an AMR annotation from AMRFinder.

| Colour | Meaning |
|---|---|
| Orange | *Persistent* gene (core genome) |
| Green | *Shell* gene (intermediate frequency) |
| Blue | *Cloud* gene (rare / strain-specific) |
| **Red outer ring** | AMR-annotated gene (hover to display element symbol, drug class, and identity) |

The AMR track provides an immediate view of which chromosomal regions concentrate resistance genes. The inner RGP arcs delimit the extent of each detected genomic island.


In [ ]:
import json
from pathlib import Path
from IPython.display import display, HTML

matches = list(Path("projection_output").rglob("input_genome_proksee.json"))

with open(matches[0]) as ff:
    data = json.load(ff)

# Add a legend entry for the AMR track
data['cgview']['legend']['items'].append({
    'decoration': 'arrow',
    'name': 'AMR',
    'swatchColor': '#e74c3c'
})

# Add a dedicated AMR track as a second outside ring (thinner than the Gene ring)
data['cgview']['tracks'].append({
    'dataKeys': 'AMR',
    'dataMethod': 'source',
    'dataType': 'feature',
    'name': 'AMR',
    'position': 'outside',
    'separateFeaturesBy': 'strand',
    'thicknessRatio': 0.5
})

# Duplicate each AMR-annotated gene as a new feature assigned to the AMR track
for feature in list(data['cgview']['features']):
    meta = feature.get('meta', {})
    if any(k.startswith('family_amrfinder_') for k in meta):
        symbol = meta.get('family_amrfinder_Element_symbol', ['AMR'])
        name = symbol[0] if isinstance(symbol, list) else symbol
        data['cgview']['features'].append({
            'contig':  feature['contig'],
            'legend':  'AMR',
            'meta':    meta,
            'name':    name,
            'source':  'AMR',
            'start':   feature['start'],
            'stop':    feature['stop'],
            'strand':  feature.get('strand', 1),
            'type':    'AMR'
        })


out_path = matches[0].stem + "_amr.json"
with open(out_path, "w") as _f:
    json.dump(data, _f)
print(f"Saved {out_path}")
json_str = json.dumps(data)
container_id = "cgview-container"

html = f"""
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/cgview/dist/cgview.css">

<div id="{container_id}" style="width:800px; height:600px; border:1px solid #ccc;">
    Loading CGView...
</div>

<script>
(function() {{
    function loadScript(src) {{
        return new Promise(function(resolve, reject) {{
            var s = document.createElement('script');
            s.src = src;
            s.onload = resolve;
            s.onerror = reject;
            document.head.appendChild(s);
        }});
    }}

    Promise.resolve()
        .then(function() {{ return loadScript('https://cdn.jsdelivr.net/npm/d3@7'); }})
        .then(function() {{ return loadScript('https://cdn.jsdelivr.net/npm/cgview/dist/cgview.min.js'); }})
        .then(function() {{
            var jsonData = {json_str};
            var container = document.getElementById('{container_id}');
            container.innerHTML = '';
            var viewer = new CGView.Viewer('{container_id}', {{ width: 800, height: 600 }});
            viewer.io.loadJSON(jsonData);
        }})
        .catch(function(err) {{
            document.getElementById('{container_id}').innerText = 'Failed to load CGView: ' + err;
        }});
}})();
</script>
"""

display(HTML(html))


## AMR Gene Content per RGP

The table below lists all RGPs that carry at least one AMR gene family in strain ABO21-A003, parsed from the projection GFF. Each row shows the number of AMR genes, the drug classes represented, and the specific gene symbols, together with the spot assignment.

The RGP at **spot_31** (CP136181.1_RGP_5) carries the highest AMR load, with multiple copies of *bla*TEM-1 (beta-lactam resistance) and *aph*A1 (kanamycin resistance) consistent with the ABGRI2 resistance island ([André et al., 2024](https://www.microbiologyresearch.org/content/journal/mgen/10.1099/mgen.0.001292)).

In [ ]:
from collections import Counter
from pygenomeviz.parser import Gff

gff = Gff("projection_output/input_genome/input_genome.gff")

rows = []
for rec in gff.all_records:
    if rec.type != "CDS":
        continue
    rgp = rec.attrs.get("rgp", [None])[0]
    if rgp is None or rec.attrs.get("family_amrfinder_Type", [None])[0] != "AMR":
        continue
    rows.append({
        "RGP":    rgp,
        "symbol": rec.attrs.get("family_amrfinder_Element_symbol", [""])[0],
        "cls":    rec.attrs.get("family_amrfinder_Class", [""])[0],
    })

df_rgp_amr = (
    pd.DataFrame(rows)
    .groupby("RGP")
    .agg(
        **{
            "nb AMR genes": ("symbol", "count"),
            "Drug classes": ("cls", lambda x: ", ".join(sorted(x.unique()))),
            "AMR genes": (
                "symbol",
                lambda x: ", ".join(
                    f"{count} × {gene}" if count > 1 else gene
                    for gene, count in sorted(Counter(x).items())
                )
            ),
        }
    )
    .reset_index()
    .merge(
        df_proj_spots.rename(columns={"region": "RGP"}),
        on="RGP",
        how="left"
    )
    .sort_values("nb AMR genes", ascending=False)
    .reset_index(drop=True)
)
df_rgp_amr["spot_id"] = df_rgp_amr["spot_id"].fillna("No spot")
display(df_rgp_amr[["RGP", "spot_id", "nb AMR genes", "Drug classes", "AMR genes"]])
